In [1]:
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from datetime import date

RAW_DIR = Path("raw_data")
CURATED_DIR = Path("curated_data")

RAW_DIR.mkdir(exist_ok=True)
CURATED_DIR.mkdir(exist_ok=True)

COLLECTION_DATE = date.today().isoformat()

print("Folders ready")
print("Collection date:", COLLECTION_DATE)

Folders ready
Collection date: 2026-06-04


In [2]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

def get_soup(url):
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

def clean_text(text):
    if pd.isna(text):
        return None
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text

In [3]:
aws_url = "https://aws.amazon.com/products/"
aws_soup = get_soup(aws_url)

aws_text = aws_soup.get_text("\n")

with open(RAW_DIR / "aws_products_page_raw.txt", "w", encoding="utf-8") as f:
    f.write(aws_text)

print("AWS page downloaded")
print(aws_text[:1000])

AWS page downloaded


























Cloud Services - Build and Scale Securely- AWS






































































































































































Skip to main content










































 
 
 
 
 


















































































































Filter: All




 


















































































English












Contact us


AWS Marketplace




Support 
 














My account 
 














































 
 
 
 
 
























Search


























































































Filter: All




 








































Sign in to console




Create account






















































































Hom

In [4]:
aws_links = []

for a in aws_soup.find_all("a", href=True):
    text = clean_text(a.get_text())
    href = a["href"]

    if text and href:
        aws_links.append({
            "text": text,
            "href": href
        })

aws_links_df = pd.DataFrame(aws_links).drop_duplicates()

aws_links_df.head(50)

,text,href
0,Skip to main content,#aws-page-content-main
1,Contact us,/contact-us/?nc2=h_ut_cu
2,AWS Marketplace,https://aws.amazon.com/marketplace?nc2=h_utmp
3,Sign in to console,https://console.aws.amazon.com/console/home/?n...
4,Create account,https://signin.aws.amazon.com/signup?request_t...
5,Home,/
6,Learn more about AWS Regions,/about-aws/global-infrastructure/?hp=tile&tile...
7,Resources Read what top analysts such as Gartn...,/resources/analyst-reports/?sc_icampaign=aware...
8,Training On-demand resources to help you devel...,/training/?sc_icampaign=aware_aws-training&sc_...
9,Partners Join AWS Partner Network to build and...,/partners/?sc_icampaign=aware_apn_recruit&sc_i...


In [5]:
aws_service_links = aws_links_df[
    aws_links_df["href"].str.contains("/products/|/what-is/|/s3/|/ec2/|/lambda/|/rds/|/sagemaker/|/bedrock/", case=False, na=False)
].copy()

aws_service_links = aws_service_links[
    ~aws_service_links["text"].str.contains(
        "Sign in|Create account|Contact us|Support|Marketplace|English|Learn more|Documentation",
        case=False,
        na=False
    )
]

aws_service_links = aws_service_links.drop_duplicates(subset=["text"])

aws_service_links.head(100)

,text,href
15,What Is Agentic AI?,/what-is/agentic-ai/?nc1=f_cc
16,Cloud Computing Concepts Hub,/what-is/?nc1=f_cc


In [6]:
aws_service_links.to_csv(
    RAW_DIR / "aws_service_links_candidate.csv",
    index=False
)

print("Candidate AWS links:", len(aws_service_links))
aws_service_links[["text", "href"]].head(100)

Candidate AWS links: 2


,text,href
15,What Is Agentic AI?,/what-is/agentic-ai/?nc1=f_cc
16,Cloud Computing Concepts Hub,/what-is/?nc1=f_cc


In [7]:
import pandas as pd

native = pd.read_csv("native_cloud_service_catalog.csv")

aws = native[native["provider"]=="AWS"]

print("AWS services:", len(aws))

aws["service_name"].sort_values().tolist()[:100]

AWS services: 69


['AMSOperations',
 'AWSBackup',
 'AWSBillingConductor',
 'AWSCloudTrail',
 'AWSCodeArtifact',
 'AWSCodePipeline',
 'AWSComputeOptimizer',
 'AWSDatabaseMigrationSvc',
 'AWSDeveloperSupport',
 'AWSElementalMediaTailor',
 'AWSEndUserMessaging3pFees',
 'AWSEvents',
 'AWSIAMAccessAnalyzer',
 'AWSInterconnect',
 'AWSIoTAnalytics',
 'AWSIoTEvents',
 'AWSLambda',
 'AWSNetworkFirewall',
 'AWSQueueService',
 'AWSSecurityHub',
 'AWSSecurityIncidentResponse',
 'AWSStorageGateway',
 'AWSStorageGatewayDeepArchive',
 'AWSSupplyChain',
 'AWSSupportBusiness',
 'AWSSupportEnterprise',
 'AWSSupportEssential',
 'AWSTelcoNetworkBuilder',
 'AWSUnifiedOperations',
 'AmazonApiGateway',
 'AmazonAthena',
 'AmazonBedrock',
 'AmazonBedrockAgentCore',
 'AmazonBedrockFoundationModels',
 'AmazonBedrockService',
 'AmazonDevOpsGuru',
 'AmazonDynamoDB',
 'AmazonEC2',
 'AmazonEC2OCPULicenseFees',
 'AmazonECS',
 'AmazonEFS',
 'AmazonEKS',
 'AmazonEKSAnywhere',
 'AmazonFSx',
 'AmazonGuardDuty',
 'AmazonKinesis',
 'AmazonK

In [8]:
aws.groupby("category").size().sort_values(ascending=False)

category
AI / Machine Learning      14
Analytics                   8
Compute                     8
Security                    7
Storage                     6
Billing / Support           5
DevOps                      5
Networking                  5
Messaging / Integration     5
Management                  3
Database                    3
dtype: int64

In [9]:
native = pd.read_csv("native_cloud_service_catalog.csv")

azure = native[native["provider"]=="Azure"]

print("Azure services:", len(azure))

azure["service_name"].sort_values().tolist()

Azure services: 32


['Application Gateway',
 'Archive Storage',
 'Azure AI Search',
 'Azure AI Services',
 'Azure Automation',
 'Azure CDN',
 'Azure Cache for Redis',
 'Azure Container Instances',
 'Azure DNS',
 'Azure Database for MySQL',
 'Azure Database for PostgreSQL',
 'Azure Firewall',
 'Azure Functions',
 'Azure Kubernetes Service',
 'Azure Machine Learning',
 'Azure Monitor',
 'Azure OpenAI Service',
 'Azure Policy',
 'Azure Resource Manager',
 'Azure SQL Database',
 'Azure Stream Analytics',
 'Azure Synapse Analytics',
 'Blob Storage',
 'Data Lake Storage',
 'Disk Storage',
 'Load Balancer',
 'Microsoft Defender for Cloud',
 'Microsoft Sentinel',
 'Queue Storage',
 'VPN Gateway',
 'Virtual Machines',
 'Virtual Network']

In [10]:
gcp = native[native["provider"]=="GCP"]

print("GCP services:", len(gcp))

gcp["service_name"].sort_values().tolist()[:100]

GCP services: 359


[' Secured Postgresql on Windows 2012 R2',
 'AI/ML development, training & inference using Python & Jupyter',
 'AISE PyTorch CPU Production',
 'AISE PyTorch NVidia GPU Production',
 'AISE TensorFlow NVidia GPU Production',
 'APEX Protection Storage for Google (DDVE)',
 'API Gateway',
 'AXI AI Forecast ML',
 'AXI AI SAP EasyConnect',
 'AXI AI SAP EasyConnect Plus',
 'Abzooba Inc. xpresso-ai',
 'Accelerated Joomla',
 'Active Directory Domain Controller 2016',
 'Active Directory Domain Controller 2019',
 'Airlock Gateway 7.4',
 'Airlock Gateway 7.5',
 'Airlock Gateway 7.6',
 'Airlock Gateway 7.7',
 'Airtable',
 'Aiven Platform: Managed Database and Data Streaming Services',
 'Aiven for Apache Cassandra',
 'Aiven for Apache Kafka',
 'Aiven for Apache Kafka Connect',
 'Aiven for Apache Kafka MirrorMaker 2',
 'Aiven for MySQL',
 'Aiven for PostgreSQL',
 'Aiven for Redis',
 'Anthos Policy Controller',
 'Aparavi Aparavi Active Archive',
 'App Engine',
 'Aqua Cloud Native Security Platform',
 '

In [11]:
import pandas as pd

native = pd.read_csv("native_cloud_service_catalog.csv")

gcp = native[native["provider"]=="GCP"].copy()

marketplace_keywords = [
    "Cisco",
    "Fortinet",
    "Forti",
    "Aiven",
    "Jenkins",
    "CloudBees",
    "MediaWiki",
    "Airtable",
    "Aqua",
    "Barracuda",
    "BlueCat",
    "Check Point",
    "Commvault",
    "Cohesity",
    "DataStax",
    "CloudGuard",
    "VMware",
    "Redis VM",
    "Domain Controller",
    "BYOL"
]

In [12]:
pattern = "|".join(marketplace_keywords)

gcp_native = gcp[
    ~gcp["service_name"].str.contains(
        pattern,
        case=False,
        na=False
    )
].copy()

print("Original GCP:", len(gcp))
print("Clean GCP:", len(gcp_native))

Original GCP: 359
Clean GCP: 293


In [13]:
aws = native[native["provider"]=="AWS"]
azure = native[native["provider"]=="Azure"]

native_v2 = pd.concat(
    [
        aws,
        azure,
        gcp_native
    ],
    ignore_index=True
)

native_v2.to_csv(
    "curated_data/native_cloud_service_catalog_v2.csv",
    index=False
)

native_v2.groupby("provider")["service_name"].nunique()

provider
AWS       69
Azure     32
GCP      287
Name: service_name, dtype: int64

In [14]:
native_v2.groupby("provider")["service_name"].nunique()

provider
AWS       69
Azure     32
GCP      287
Name: service_name, dtype: int64

In [15]:
gcp_native["service_name"].sort_values().tolist()[:200]

[' Secured Postgresql on Windows 2012 R2',
 'AI/ML development, training & inference using Python & Jupyter',
 'AISE PyTorch CPU Production',
 'AISE PyTorch NVidia GPU Production',
 'AISE TensorFlow NVidia GPU Production',
 'APEX Protection Storage for Google (DDVE)',
 'API Gateway',
 'AXI AI Forecast ML',
 'AXI AI SAP EasyConnect',
 'AXI AI SAP EasyConnect Plus',
 'Abzooba Inc. xpresso-ai',
 'Accelerated Joomla',
 'Airlock Gateway 7.4',
 'Airlock Gateway 7.5',
 'Airlock Gateway 7.6',
 'Airlock Gateway 7.7',
 'Anthos Policy Controller',
 'Aparavi Aparavi Active Archive',
 'App Engine',
 'Arista Networks Inc. cloudeos-router-payg',
 'Artifact Registry',
 'AtScale Inc AtScale Adaptive Analytics',
 'AtomicWP Cloud Workload Security',
 'AtomicWP Cloud Workload Security for Docker',
 'Atos Database hotel for GCP',
 'AutoML Neuton',
 'Aviatrix Cloud Network CoPilot',
 'Aviatrix Cloud Network Controller',
 'Backup and DR Service',
 'Backup for GKE',
 'BeyondInsight (SQL-Free)',
 'BigQuery',
 

In [16]:
services = sorted(gcp_native["service_name"].unique())

for s in services[:200]:
    print(s)

 Secured Postgresql on Windows 2012 R2
AI/ML development, training & inference using Python & Jupyter
AISE PyTorch CPU Production
AISE PyTorch NVidia GPU Production
AISE TensorFlow NVidia GPU Production
APEX Protection Storage for Google (DDVE)
API Gateway
AXI AI Forecast ML
AXI AI SAP EasyConnect
AXI AI SAP EasyConnect Plus
Abzooba Inc. xpresso-ai
Accelerated Joomla
Airlock Gateway 7.4
Airlock Gateway 7.5
Airlock Gateway 7.6
Airlock Gateway 7.7
Anthos Policy Controller
Aparavi Aparavi Active Archive
App Engine
Arista Networks Inc. cloudeos-router-payg
Artifact Registry
AtScale Inc AtScale Adaptive Analytics
AtomicWP Cloud Workload Security
AtomicWP Cloud Workload Security for Docker
Atos Database hotel for GCP
AutoML Neuton
Aviatrix Cloud Network CoPilot
Aviatrix Cloud Network Controller
Backup and DR Service
Backup for GKE
BeyondInsight (SQL-Free)
BigQuery
BigQuery BI Engine
BigQuery Data Transfer Service
BigQuery Reservation API
BigQuery Storage API
BizStats AI
Blockchain Accelerato

In [17]:
gcp_native_keywords = [
    "BigQuery",
    "Cloud ",
    "Compute Engine",
    "Dataproc",
    "Dataproc Metastore",
    "Artifact Registry",
    "App Engine",
    "Anthos",
    "Identity Platform",
    "Eventarc",
    "Backup for GKE",
    "Backup and DR Service",
    "Database Migration",
    "Kubernetes Engine",
    "Network Connectivity Center",
    "Network Security",
    "Pub/Sub Lite",
    "Duet AI",
    "Firebase",
    "Google Maps Platform",
]

In [18]:
pattern = "|".join(gcp_native_keywords)

gcp_official = gcp[
    gcp["service_name"].str.contains(
        pattern,
        case=False,
        na=False
    )
].copy()

print("Official GCP Services:", len(gcp_official))

Official GCP Services: 79


In [19]:
sorted(gcp_official["service_name"].unique())

['Anthos Policy Controller',
 'App Engine',
 'Aqua Cloud Native Security Platform',
 'Artifact Registry',
 'AtomicWP Cloud Workload Security',
 'AtomicWP Cloud Workload Security for Docker',
 'Aviatrix Cloud Network CoPilot',
 'Aviatrix Cloud Network Controller',
 'Backup and DR Service',
 'Backup for GKE',
 'BigQuery',
 'BigQuery BI Engine',
 'BigQuery Data Transfer Service',
 'BigQuery Reservation API',
 'BigQuery Storage API',
 'BlueCat DNS for Google Cloud Platform',
 'Cavirin First Line of Defense for Cloud Security - BYOL',
 'Check Point CloudGuard Network Security Autoscaling (BYOL)',
 'Check Point CloudGuard Network Security High Availability (BYOL)',
 'Cloud AutoML',
 'Cloud Bigtable',
 'Cloud Build',
 'Cloud Contact Center AI Platform',
 'Cloud DNS',
 'Cloud Data Loss Prevention',
 'Cloud Dataflow',
 'Cloud Document AI API',
 'Cloud Domains',
 'Cloud KMS KACLS',
 'Cloud Key Management Service (KMS)',
 'Cloud Logging',
 'Cloud Machine Learning Engine',
 'Cloud Memorystore for 

In [20]:
official_gcp_services = [
    "Anthos Policy Controller",
    "App Engine",
    "Artifact Registry",
    "Backup and DR Service",
    "Backup for GKE",
    "BigQuery",
    "BigQuery BI Engine",
    "BigQuery Data Transfer Service",
    "BigQuery Reservation API",
    "BigQuery Storage API",
    "Cloud AutoML",
    "Cloud Bigtable",
    "Cloud Build",
    "Cloud Contact Center AI Platform",
    "Cloud DNS",
    "Cloud Data Loss Prevention",
    "Cloud Dataflow",
    "Cloud Document AI API",
    "Cloud Domains",
    "Cloud Key Management Service (KMS)",
    "Cloud Logging",
    "Cloud Machine Learning Engine",
    "Cloud Memorystore for Redis",
    "Cloud Monitoring",
    "Cloud Pub/Sub",
    "Cloud Run",
    "Cloud Run Functions",
    "Cloud SQL",
    "Cloud Spanner",
    "Cloud Speech API",
    "Cloud Storage",
    "Cloud Text-to-Speech API",
    "Cloud Vision API",
    "Compute Engine",
    "Database Migration",
    "Dataproc",
    "Dataproc Metastore",
    "Duet AI",
    "Eventarc",
    "Firebase Realtime Database",
    "Google Maps Platform Air Quality Service",
    "Identity Platform",
    "Kubernetes Engine",
    "Network Connectivity Center",
    "Network Security",
    "Pub/Sub Lite"
]

In [21]:
gcp_final = gcp[
    gcp["service_name"].isin(official_gcp_services)
].copy()

print("Official GCP Services:", gcp_final["service_name"].nunique())

sorted(gcp_final["service_name"].unique())

Official GCP Services: 46


['Anthos Policy Controller',
 'App Engine',
 'Artifact Registry',
 'Backup and DR Service',
 'Backup for GKE',
 'BigQuery',
 'BigQuery BI Engine',
 'BigQuery Data Transfer Service',
 'BigQuery Reservation API',
 'BigQuery Storage API',
 'Cloud AutoML',
 'Cloud Bigtable',
 'Cloud Build',
 'Cloud Contact Center AI Platform',
 'Cloud DNS',
 'Cloud Data Loss Prevention',
 'Cloud Dataflow',
 'Cloud Document AI API',
 'Cloud Domains',
 'Cloud Key Management Service (KMS)',
 'Cloud Logging',
 'Cloud Machine Learning Engine',
 'Cloud Memorystore for Redis',
 'Cloud Monitoring',
 'Cloud Pub/Sub',
 'Cloud Run',
 'Cloud Run Functions',
 'Cloud SQL',
 'Cloud Spanner',
 'Cloud Speech API',
 'Cloud Storage',
 'Cloud Text-to-Speech API',
 'Cloud Vision API',
 'Compute Engine',
 'Database Migration',
 'Dataproc',
 'Dataproc Metastore',
 'Duet AI',
 'Eventarc',
 'Firebase Realtime Database',
 'Google Maps Platform Air Quality Service',
 'Identity Platform',
 'Kubernetes Engine',
 'Network Connectivity 

In [22]:
aws = native[native["provider"]=="AWS"]
azure = native[native["provider"]=="Azure"]

native_v3 = pd.concat(
    [aws, azure, gcp_final],
    ignore_index=True
)

native_v3.to_csv(
    "curated_data/native_cloud_service_catalog_v3.csv",
    index=False
)

native_v3.groupby("provider")["service_name"].nunique()

provider
AWS      69
Azure    32
GCP      46
Name: service_name, dtype: int64

In [23]:
native_v3.groupby("provider")["service_name"].nunique()

provider
AWS      69
Azure    32
GCP      46
Name: service_name, dtype: int64

In [24]:
ai = pd.read_csv("native_ai_services_enriched.csv")

ai[[
    "provider",
    "service_name",
    "ai_type"
]]

,provider,service_name,ai_type
0,AWS,AmazonLookoutVision,Computer Vision
1,AWS,AmazonBedrockService,Foundation Models
2,AWS,AmazonSageMaker,ML Platform
3,AWS,AmazonBedrock,Foundation Models
4,AWS,AmazonBedrockAgentCore,Foundation Models
5,AWS,AmazonBedrockFoundationModels,Foundation Models
6,Azure,Azure OpenAI Service,Foundation Models
7,Azure,Azure Machine Learning,ML Platform
8,Azure,Azure AI Search,General AI
9,Azure,Azure AI Services,General AI


In [25]:
native_v3.to_csv(
    "curated_data/native_cloud_service_catalog_v3.csv",
    index=False
)

In [26]:
import os

os.listdir("curated_data")

['native_cloud_service_catalog_v2.csv', 'native_cloud_service_catalog_v3.csv']

In [27]:
ai = pd.read_csv("native_ai_services_enriched.csv")

ai[[
    "provider",
    "service_name",
    "ai_type"
]]

,provider,service_name,ai_type
0,AWS,AmazonLookoutVision,Computer Vision
1,AWS,AmazonBedrockService,Foundation Models
2,AWS,AmazonSageMaker,ML Platform
3,AWS,AmazonBedrock,Foundation Models
4,AWS,AmazonBedrockAgentCore,Foundation Models
5,AWS,AmazonBedrockFoundationModels,Foundation Models
6,Azure,Azure OpenAI Service,Foundation Models
7,Azure,Azure Machine Learning,ML Platform
8,Azure,Azure AI Search,General AI
9,Azure,Azure AI Services,General AI


In [28]:
import pandas as pd

ai = pd.read_csv("native_ai_services_enriched.csv")

# 刪除第三方
ai = ai[
    ai["service_name"] != "Dace IT℠with Sense Traffic Pulse™ and Vertex AI"
].copy()

In [29]:
taxonomy_map = {
    
    # AWS
    "AmazonBedrock":"Foundation Models",
    "AmazonBedrockService":"Foundation Models",
    "AmazonBedrockAgentCore":"AI Agents",
    "AmazonBedrockFoundationModels":"Foundation Models",
    "AmazonSageMaker":"MLOps / ML Platform",
    "AmazonLookoutVision":"Computer Vision",

    # Azure
    "Azure OpenAI Service":"Foundation Models",
    "Azure Machine Learning":"MLOps / ML Platform",
    "Azure AI Search":"Vector Search / RAG",
    "Azure AI Services":"General AI",

    # GCP
    "Vertex AI":"MLOps / ML Platform",
    "Vertex AI Search":"Vector Search / RAG",
    "Vertex AI Vision":"Computer Vision",
    "Cloud Document AI API":"Document AI",
    "Cloud Contact Center AI Platform":"Conversational AI",
    "Cloud Machine Learning Engine":"MLOps / ML Platform"
}

In [30]:
ai["ai_taxonomy"] = ai["service_name"].map(taxonomy_map)

ai[[
    "provider",
    "service_name",
    "ai_taxonomy"
]]

,provider,service_name,ai_taxonomy
0,AWS,AmazonLookoutVision,Computer Vision
1,AWS,AmazonBedrockService,Foundation Models
2,AWS,AmazonSageMaker,MLOps / ML Platform
3,AWS,AmazonBedrock,Foundation Models
4,AWS,AmazonBedrockAgentCore,AI Agents
5,AWS,AmazonBedrockFoundationModels,Foundation Models
6,Azure,Azure OpenAI Service,Foundation Models
7,Azure,Azure Machine Learning,MLOps / ML Platform
8,Azure,Azure AI Search,Vector Search / RAG
9,Azure,Azure AI Services,General AI


In [31]:
ai.groupby(
    ["provider","ai_taxonomy"]
).size()

provider  ai_taxonomy        
AWS       AI Agents              1
          Computer Vision        1
          Foundation Models      3
          MLOps / ML Platform    1
Azure     Foundation Models      1
          General AI             1
          MLOps / ML Platform    1
          Vector Search / RAG    1
GCP       Computer Vision        1
          Conversational AI      1
          Document AI            1
          MLOps / ML Platform    2
          Vector Search / RAG    1
dtype: int64

In [32]:
ai_taxonomy_summary = (
    ai.groupby(
        ["provider","ai_taxonomy"]
    )
    .size()
    .reset_index(name="service_count")
)

ai_taxonomy_summary.to_csv(
    "curated_data/ai_taxonomy_summary_v2.csv",
    index=False
)

ai.to_csv(
    "curated_data/ai_services_v2.csv",
    index=False
)

ai_taxonomy_summary

,provider,ai_taxonomy,service_count
0,AWS,AI Agents,1
1,AWS,Computer Vision,1
2,AWS,Foundation Models,3
3,AWS,MLOps / ML Platform,1
4,Azure,Foundation Models,1
5,Azure,General AI,1
6,Azure,MLOps / ML Platform,1
7,Azure,Vector Search / RAG,1
8,GCP,Computer Vision,1
9,GCP,Conversational AI,1


In [33]:
ai_taxonomy_summary

,provider,ai_taxonomy,service_count
0,AWS,AI Agents,1
1,AWS,Computer Vision,1
2,AWS,Foundation Models,3
3,AWS,MLOps / ML Platform,1
4,Azure,Foundation Models,1
5,Azure,General AI,1
6,Azure,MLOps / ML Platform,1
7,Azure,Vector Search / RAG,1
8,GCP,Computer Vision,1
9,GCP,Conversational AI,1
